In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)


In [ ]:
from adapters import ShareGPTAdapter, AlpacaAdapter, MultiTurnAdapter

sample = next(iter(ShareGPTAdapter("data/chatalpaca-20k.json", "chatalpaca").records()))
print("ChatAlpaca sample turns:", len(sample.turns))
for t in sample.turns[:4]:
    print(f"  {t.role}: {t.content[:80]}")


In [ ]:
!python scripts/makeJson.py


In [ ]:
import json
with open("data/clean/sft.jsonl", "r", encoding="utf-8") as f:
    first = json.loads(next(f))
print("Total turns in first record:", len(first["turns"]))
print(first["text"][:500])


## Train tokenizer


In [ ]:
!python scripts/trainTokenizer.py --vocab-size 8000


## Train model


In [ ]:
!python train/train.py --max-steps 3000


In [ ]:
import torch
from train.chat import load_model, generate
from train.data import load_tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model, cfg, tokenizer_rel = load_model("models/buddyGPT.pt", device)
tokenizer = load_tokenizer(tokenizer_rel)

user_tag = tokenizer.token_to_id("<|user|>")
asst_tag = tokenizer.token_to_id("<|assistant|>")
end_id   = tokenizer.token_to_id("<|end|>")

def ask(message: str) -> str:
    body = tokenizer.encode(message).ids
    prompt = (
        [user_tag] + tokenizer.encode("\n").ids
        + body + tokenizer.encode("\n").ids
        + [asst_tag] + tokenizer.encode("\n").ids
    )
    return generate(model, tokenizer, prompt, end_id, device=device)

print("You: Hello")
print("AI:", ask("Hello"))
print()
print("You: What is Python?")
print("AI:", ask("What is Python?"))
